In [0]:
USE CATALOG smart_odoo;
 
-- =====================================================
-- FACT: gold.fact_gl_transactions
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.fact_gl_transactions
(
    move_line_id INT,
 
    account_id INT,
    account_name STRING,
 
    journal_id INT,
    journal_name STRING,
 
    partner_id INT,
    partner_name STRING,
 
    product_id INT,
    product_name STRING,
 
    company_id INT,
    company_name STRING,
 
    move_name STRING,
    parent_state STRING,
    ref STRING,
    display_type STRING,
 
    debit DOUBLE,
    credit DOUBLE,
    balance DOUBLE,
    amount_currency DOUBLE,
    tax_base_amount DOUBLE,
    amount_residual DOUBLE,
 
    move_date DATE,
    date_maturity DATE,
 
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.fact_gl_transactions AS target
 
USING
(
    SELECT
        l.account_move_line_id  AS move_line_id,
 
        l.account_id,
        l.account_name,
 
        l.journal_id,
        l.journal_name,
 
        l.partner_id,
        l.partner_name,
 
        l.product_id,
        l.product_name,
 
        l.company_id,
        l.company_name,
 
        l.move_name,
        l.parent_state,
        l.ref,
        l.display_type,
 
        l.debit,
        l.credit,
        l.balance,
        l.amount_currency,
        l.tax_base_amount,
        l.amount_residual,
 
        CAST(l.date AS DATE)          AS move_date,
        CAST(l.date_maturity AS DATE) AS date_maturity,
 
        l.created_at,
        l.updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.account_move_line l
 
    WHERE l.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.fact_gl_transactions),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.move_line_id = source.move_line_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.account_id       = source.account_id,
    target.account_name     = source.account_name,
    target.journal_id       = source.journal_id,
    target.journal_name     = source.journal_name,
    target.partner_id       = source.partner_id,
    target.partner_name     = source.partner_name,
    target.product_id       = source.product_id,
    target.product_name     = source.product_name,
    target.company_id       = source.company_id,
    target.company_name     = source.company_name,
    target.move_name        = source.move_name,
    target.parent_state     = source.parent_state,
    target.ref              = source.ref,
    target.display_type     = source.display_type,
    target.debit            = source.debit,
    target.credit           = source.credit,
    target.balance          = source.balance,
    target.amount_currency  = source.amount_currency,
    target.tax_base_amount  = source.tax_base_amount,
    target.amount_residual  = source.amount_residual,
    target.move_date        = source.move_date,
    target.date_maturity    = source.date_maturity,
    target.updated_at       = source.updated_at,
    target.dwh_loaded_at    = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 


In [0]:
-- =====================================================
-- FACT: gold.fact_invoice
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.fact_invoice
(
    move_id INT,
 
    partner_id INT,
    partner_name STRING,
 
    company_id INT,
    company_name STRING,
 
    journal_id INT,
    journal_name STRING,
 
    invoice_user_id INT,
    invoice_user_name STRING,
 
    currency_id INT,
    currency_name STRING,
 
    reversed_entry_id INT,
 
    name STRING,
    ref STRING,
    move_type STRING,
    state STRING,
    payment_state STRING,
    payment_reference STRING,
    invoice_origin STRING,
    narration STRING,
 
    invoice_currency_rate DOUBLE,
    amount_untaxed DOUBLE,
    amount_tax DOUBLE,
    amount_total DOUBLE,
    amount_residual DOUBLE,
    amount_untaxed_signed DOUBLE,
    amount_tax_signed DOUBLE,
    amount_total_signed DOUBLE,
    amount_residual_signed DOUBLE,
 
    invoice_status STRING,
 
    is_move_sent BOOLEAN,
    posted_before BOOLEAN,
 
    move_date DATE,
    invoice_date DATE,
    invoice_date_due DATE,
    delivery_date DATE,
 
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.fact_invoice AS target
 
USING
(
    SELECT
        m.account_move_id           AS move_id,
 
        m.partner_id,
        m.partner_name,
 
        m.company_id,
        m.company_name,
 
        m.journal_id,
        m.journal_name,
 
        m.invoice_user_id,
        m.invoice_user_name,
 
        m.currency_id,
        m.currency_name,
 
        m.reversed_entry_id,
 
        m.name,
        m.ref,
        m.move_type,
        m.state,
        m.payment_state,
        m.payment_reference,
        m.invoice_origin,
        m.narration,
 
        m.invoice_currency_rate,
        m.amount_untaxed,
        m.amount_tax,
        m.amount_total,
        m.amount_residual,
        m.amount_untaxed_signed,
        m.amount_tax_signed,
        m.amount_total_signed,
        m.amount_residual_signed,
 
        CASE
            WHEN m.amount_residual = 0                                              THEN 'Paid'
            WHEN m.amount_residual > 0 AND m.invoice_date_due < current_date()     THEN 'Overdue'
            ELSE 'Open'
        END AS invoice_status,
 
        m.is_move_sent,
        m.posted_before,
 
        CAST(m.date AS DATE)              AS move_date,
        CAST(m.invoice_date AS DATE)      AS invoice_date,
        CAST(m.invoice_date_due AS DATE)  AS invoice_date_due,
        CAST(m.delivery_date AS DATE)     AS delivery_date,
 
        m.created_at,
        m.updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.account_move m
 
    WHERE m.move_type IN ('out_invoice', 'in_invoice', 'out_refund', 'in_refund')
    AND m.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.fact_invoice),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.move_id = source.move_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.partner_id               = source.partner_id,
    target.partner_name             = source.partner_name,
    target.company_id               = source.company_id,
    target.company_name             = source.company_name,
    target.journal_id               = source.journal_id,
    target.journal_name             = source.journal_name,
    target.invoice_user_id          = source.invoice_user_id,
    target.invoice_user_name        = source.invoice_user_name,
    target.currency_id              = source.currency_id,
    target.currency_name            = source.currency_name,
    target.reversed_entry_id        = source.reversed_entry_id,
    target.name                     = source.name,
    target.ref                      = source.ref,
    target.move_type                = source.move_type,
    target.state                    = source.state,
    target.payment_state            = source.payment_state,
    target.payment_reference        = source.payment_reference,
    target.invoice_origin           = source.invoice_origin,
    target.narration                = source.narration,
    target.invoice_currency_rate    = source.invoice_currency_rate,
    target.amount_untaxed           = source.amount_untaxed,
    target.amount_tax               = source.amount_tax,
    target.amount_total             = source.amount_total,
    target.amount_residual          = source.amount_residual,
    target.amount_untaxed_signed    = source.amount_untaxed_signed,
    target.amount_tax_signed        = source.amount_tax_signed,
    target.amount_total_signed      = source.amount_total_signed,
    target.amount_residual_signed   = source.amount_residual_signed,
    target.invoice_status           = source.invoice_status,
    target.is_move_sent             = source.is_move_sent,
    target.posted_before            = source.posted_before,
    target.move_date                = source.move_date,
    target.invoice_date             = source.invoice_date,
    target.invoice_date_due         = source.invoice_date_due,
    target.delivery_date            = source.delivery_date,
    target.updated_at               = source.updated_at,
    target.dwh_loaded_at            = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 

In [0]:
-- =====================================================
-- DIM: gold.dim_journal
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.dim_journal
(
    journal_id INT,
 
    company_id INT,
    company_name STRING,
 
    journal_code STRING,
    journal_name STRING,
    journal_type STRING,
 
    bank_statements_source STRING,
    invoice_reference_type STRING,
    invoice_reference_model STRING,
 
    is_active BOOLEAN,
    show_on_dashboard BOOLEAN,
    refund_sequence BOOLEAN,
    payment_sequence BOOLEAN,
 
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.dim_journal AS target
 
USING
(
    SELECT
        account_journal_id      AS journal_id,
        company_id,
        company_name,
        code                    AS journal_code,
        name                    AS journal_name,
        type                    AS journal_type,
        bank_statements_source,
        invoice_reference_type,
        invoice_reference_model,
        active                  AS is_active,
        show_on_dashboard,
        refund_sequence,
        payment_sequence,
        created_at,
        updated_at,
        current_timestamp()     AS dwh_loaded_at
    FROM silver.account_journal
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_journal),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.journal_id = source.journal_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.company_id               = source.company_id,
    target.company_name             = source.company_name,
    target.journal_code             = source.journal_code,
    target.journal_name             = source.journal_name,
    target.journal_type             = source.journal_type,
    target.bank_statements_source   = source.bank_statements_source,
    target.invoice_reference_type   = source.invoice_reference_type,
    target.invoice_reference_model  = source.invoice_reference_model,
    target.is_active                = source.is_active,
    target.show_on_dashboard        = source.show_on_dashboard,
    target.refund_sequence          = source.refund_sequence,
    target.payment_sequence         = source.payment_sequence,
    target.updated_at               = source.updated_at,
    target.dwh_loaded_at            = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;


In [0]:
-- =====================================================
-- DIM: gold.dim_account
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.dim_account
(
    account_id INT,
 
    currency_id INT,
    currency_name STRING,
 
    account_code STRING,
    account_name STRING,
    account_type STRING,
    note STRING,
 
    is_active BOOLEAN,
    is_reconcilable BOOLEAN,
    non_trade BOOLEAN,
 
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.dim_account AS target
 
USING
(
    SELECT
        account_id,
        currency_id,
        currency_name,
        code_store          AS account_code,
        name                AS account_name,
        account_type,
        note,
        active              AS is_active,
        reconcile           AS is_reconcilable,
        non_trade,
        created_at,
        updated_at,
        current_timestamp() AS dwh_loaded_at
    FROM silver.account_account
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_account),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.account_id = source.account_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.currency_id      = source.currency_id,
    target.currency_name    = source.currency_name,
    target.account_code     = source.account_code,
    target.account_name     = source.account_name,
    target.account_type     = source.account_type,
    target.note             = source.note,
    target.is_active        = source.is_active,
    target.is_reconcilable  = source.is_reconcilable,
    target.non_trade        = source.non_trade,
    target.updated_at       = source.updated_at,
    target.dwh_loaded_at    = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
